# MobileNetV2 Transfer Learning for AQI Classification

In [ ]:
# Auto-detect: Colab or VS Code
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import files
    import zipfile, shutil
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/')
    # Handle both static/ folder or direct folders in zip
    if os.path.isdir('/content/static/'):
        DATA_DIR = '/content/static/'
    else:
        # Find the folder containing class subfolders
        candidates = ['good','moderate','unh','Unhealthy for Sensitive Groups']
        DATA_DIR = '/content/'
    print(f"Dataset: {DATA_DIR}")
    !ls -d {DATA_DIR}*/
else:
    DATA_DIR = './static/'
    print(f"Dataset: {os.path.abspath(DATA_DIR)}")

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train = datagen.flow_from_directory(DATA_DIR, target_size=(150,150), batch_size=32,
                                    class_mode='categorical', subset='training',
                                    classes=['good','moderate','unh','Unhealthy for Sensitive Groups'])
val = datagen.flow_from_directory(DATA_DIR, target_size=(150,150), batch_size=32,
                                  class_mode='categorical', subset='validation',
                                  classes=['good','moderate','unh','Unhealthy for Sensitive Groups'])
print("Classes:", train.class_indices)

Found 6881 images belonging to 4 classes.
Found 1718 images belonging to 4 classes.
Classes: {'good': 0, 'moderate': 1, 'unh': 2, 'Unhealthy for Sensitive Groups': 3}


In [2]:
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(150,150,3))
base.trainable = False

model = tf.keras.Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7588\2758406239.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(150,150,3))


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,108 (8.63 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model.fit(train, validation_data=val, epochs=15,
                    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)])

Epoch 1/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 71s 328ms/step - accuracy: 0.7688 - loss: 0.5657 - val_accuracy: 0.4319 - val_loss: 1.9177
Epoch 2/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 70s 323ms/step - accuracy: 0.7717 - loss: 0.5543 - val_accuracy: 0.4837 - val_loss: 1.7973
Epoch 3/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 105s 486ms/step - accuracy: 0.7795 - loss: 0.5342 - val_accuracy: 0.4057 - val_loss: 2.0294
Epoch 4/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 116s 537ms/step - accuracy: 0.7919 - loss: 0.5181 - val_accuracy: 0.4721 - val_loss: 1.7592
Epoch 5/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 71s 327ms/step - accuracy: 0.7904 - loss: 0.5201 - val_accuracy: 0.4872 - val_loss: 1.8185
Epoch 6/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 71s 327ms/step - accuracy: 0.7968 - loss: 0.5128 - val_accuracy: 0.4715 - val_loss: 1.8006
Epoch 7/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 90s 419ms/step - accuracy: 0.7986 - loss: 0.5050 - val_accuracy: 0.4889 - val_loss: 1.8177


In [ ]:
model.save('aqi_mobilenet_model.keras')
print("Model saved successfully as 'aqi_mobilenet_model.keras'")